In [0]:
print("Spark DataFrames Part - 3")

In [0]:
from pyspark.sql import functions as F

In [0]:
# Read a CSV file

apple_df = spark.read.format("csv").load("/FileStore/tables/apple_global_sales_dataset.csv")
display(apple_df)

In [0]:
apple_df = (
    spark.read.format("csv")
    .option("header", True)
    .load("/FileStore/tables/apple_global_sales_dataset.csv")
)
display(apple_df)

In [0]:
apple_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/FileStore/tables/apple_global_sales_dataset.csv")
)
display(apple_df)

In [0]:
apple_df.createOrReplaceTempView("apple_sales")

In [0]:
%sql
select * from apple_sales;

In [0]:
%sql
select count(*) as total_sales from apple_sales 
where country = 'India';

In [0]:
india_df = apple_df.filter(F.col("country") == "India")
india_df.display()

In [0]:
india_df.count()

In [0]:
india_df = apple_df.where(F.col("country") == "India")
india_df.display()

In [0]:
null_rtg_df = apple_df.filter(F.col("customer_rating").isNull())
null_rtg_df.count()

In [0]:
display(null_rtg_df)

In [0]:
# drop
apple_df = apple_df.drop("previous_device_os", "customer_segment")
apple_df.columns

In [0]:
apple_df.count()

In [0]:
# distinct
apple_df.distinct().count()

In [0]:
df = apple_df.dropDuplicates()
df.count()

In [0]:
df = apple_df.dropDuplicates(subset=["category"])
df.display()

In [0]:
# OrderBy | SortBy
df = apple_df.orderBy(F.col("country").desc())
df.display()

In [0]:
df = apple_df.sort(F.col("sale_date").asc())
df.display()

In [0]:
df = apple_df.sort(F.col("sale_date").desc())
df.display()

In [0]:
%sql

select 
-- month(sale_date) as month_num,
month,
count(*) as total_sales
from apple_sales
group by month(sale_date), month
order by month(sale_date) asc;

In [0]:
# aggregations
# -- sum, count, min, max and avg

df = (
    apple_df
    .groupBy(
        F.month(F.col("sale_date")).alias("month_num"),
        F.col("month")
    )
    .count().alias("total_sales")
    # .sum()
    .orderBy(F.col("month_num").asc())
)

display(df)


In [0]:
# aggregations
# -- sum, count, min, max and avg

df = (
    apple_df
    .groupBy(
        F.month(F.col("sale_date")).alias("month_num"),
        F.col("month")
    )
    .agg(
        F.count("*").alias("total_sales")
    )
    .orderBy(F.col("month_num").asc())
)

display(df.drop("month_num"))

In [0]:
# aggregations
# -- sum, count, min, max and avg

df = (
    apple_df
    .groupBy(
        F.month(F.col("sale_date")).alias("month_num"),
        F.col("month")
    )
    .agg(
        F.count("*").alias("total_sales"),
        F.round(F.sum(F.col("revenue_usd"))).alias("total_revenue")
    )
    .orderBy(F.col("month_num").asc())
)

display(df.drop("month_num"))